# EDA: Predicción de Aptitud Solar (`solar_aptitude_class`)

**Proyecto final, Machine Learning**
**Integrantes:** Jesús David Arévalo Montilla, Enmanuel David Díaz Molinares, Alex David Terán Meza.

Dataset base: *Global Solar Power Plants Dataset*, [cimejia/solarPV](https://github.com/cimejia/solarPV)
(Universidad de Alicante, Universidad Central del Ecuador). 58,978 registros.

Fuente de datos de este notebook: `Dataset_Mundial_Final.xlsx`.

---

## Pregunta de investigación

Dada una coordenada `(x, y)` con ciertas características de terreno y clima, ¿vale la pena invertir en
instalar paneles solares en ese sitio? Es decir, predecir la aptitud solar de un sitio nuevo a partir de
variables de terreno y clima disponibles antes de tomar la decisión de inversión.

## Variable objetivo

`solar_aptittude_class` (nombre real de la columna; el typo con doble "t" viene heredado de la fuente
original) es una variable categórica ordinal de 3 clases. Sobre las 58,978 plantas del dataset:

| Clase | n | % |
|---|---|---|
| Alta  | 44,295 | 75.10% |
| Media | 12,791 | 21.69% |
| Baja  |  1,892 |  3.21% |

Es un problema de clasificación multiclase fuertemente desbalanceado: `Baja` representa poco más del 3%
del total, con una razón Alta:Baja de 23.4 a 1. Esto condiciona las decisiones de modelado que siguen:
una métrica distinta de accuracy, ponderación o remuestreo de clases, y validación estratificada.

## Variables predictoras

Se mantienen como features únicamente variables disponibles antes de decidir invertir (terreno y clima),
nunca resultado de haber construido la planta:

`longitude`, `latitude`, `elevation`, `area`, `slope`, `slope_type`, `curvature`, `curvature_type`,
`aspect`, `aspect_type`, `dist_to_road`, `ambient_temperature`, `ghi`, `humidity`, `wind_speed`,
`wind_direction`, `dt_wind`.

## Variables excluidas (fuga de información o posteriores a la inversión)

- `solar_aptitude`, `solar_aptitude_rounded`: definen directamente el target por binning.
- `capacity`, `optimal_tilt`, `pv_potential`: resultado de haber construido y configurado la planta.
- `operational_status`: posterior a la decisión de inversión.
- `OBJECTID`, `code`, `plant_name`: identificadores sin valor predictivo.

## Variables analizadas sin incorporarse todavía como predictoras

- `size`: variable categórica (`Small` / `Medium` / `Big`). Su distribución varía con el estado
  operacional de la planta, lo que sugiere que describe en parte el proyecto ya construido y no solo el
  terreno disponible antes de invertir.
- `country`: geográficamente informativa, pero con 183 categorías; requiere decidir una codificación
  (categórica directa, one-hot o agregación por región) antes de usarse como predictora.

## Limitación del dataset

El dataset solo contiene coordenadas donde ya existe una planta solar construida; no hay ejemplos de
sitios sin planta. El modelo aprende a clasificar aptitud entre sitios donde ya se invirtió, y se
generaliza a sitios nuevos asumiendo que terreno y clima similares producen aptitud similar. Esta es una
asunción de generalización, no un hecho verificado por el dataset.

## Estructura del notebook

1. Configuración e importación de librerías
2. Carga de datos y validación de esquema
3. Inspección inicial
4. Calidad de datos
5. Análisis univariado de la variable objetivo y balance de clases
6. Análisis univariado de las variables predictoras numéricas
7. Variables categóricas (`slope_type`, `curvature_type`, `aspect_type`, `dt_wind`)
8. Análisis geográfico y sesgo de cobertura
9. Variables angulares (`aspect`, `wind_direction`)
10. Chequeo de fuga de datos
11. Análisis bivariado: predictoras vs. `solar_aptittude_class`
12. Correlación entre features y multicolinealidad (VIF)
13. Detección de outliers
14. Segmentación adicional (`size`, banda climática)
15. Conclusiones


## 1. Configuración e importación de librerías

In [ ]:
# Manipulación de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Estadística
from scipy.stats import kruskal

# Multicolinealidad
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CLASS_ORDER = ["Baja", "Media", "Alta"]  # orden ordinal: de menor a mayor aptitud


## 2. Carga de datos y validación de esquema

El dataset de trabajo es `Dataset_Mundial_Final.xlsx`, cargado directamente con `pandas.read_excel`, sin
transformación de esquema ni de formato numérico adicional.

Las columnas originales del dataset son:

`OBJECTID, code, plant_name, country, operational_status, longitude, latitude, elevation, area, size,
slope, slope_type, curvature, curvature_type, aspect, aspect_type, dist_to_road, ambient_temperature,
ghi, humidity, wind_speed, wind_direction, dt_wind, solar_aptitude, solar_aptitude_rounded,
solar_aptittude_class, capacity, optimal_tilt, pv_potential`

No se renombran columnas: se trabaja directamente con los nombres originales en inglés.

In [ ]:
DATA_PATH = "Dataset_Mundial_Final.xlsx"

df = pd.read_excel(DATA_PATH)
print(f"Datos cargados desde '{DATA_PATH}'.")

df.shape


In [ ]:
EXPECTED_COLUMNS = [
    "OBJECTID", "code", "plant_name", "country", "operational_status", "longitude", "latitude",
    "elevation", "area", "size", "slope", "slope_type", "curvature", "curvature_type", "aspect",
    "aspect_type", "dist_to_road", "ambient_temperature", "ghi", "humidity", "wind_speed",
    "wind_direction", "dt_wind", "solar_aptitude", "solar_aptitude_rounded",
    "solar_aptittude_class", "capacity", "optimal_tilt", "pv_potential",
]

faltantes = [c for c in EXPECTED_COLUMNS if c not in df.columns]
sobrantes = [c for c in df.columns if c not in EXPECTED_COLUMNS]

if not faltantes and not sobrantes:
    print("Esquema validado: las columnas esperadas están presentes y no hay columnas adicionales.")
else:
    if faltantes:
        print("Columnas esperadas que faltan:", faltantes)
    if sobrantes:
        print("Columnas presentes no documentadas en el esquema esperado:", sobrantes)

df.head()


In [ ]:
TARGET_COL = "solar_aptittude_class"  # nombre real de la columna (typo heredado del dataset fuente)

FEATURE_COLS = [
    "longitude", "latitude", "elevation", "area", "slope", "slope_type", "curvature",
    "curvature_type", "aspect", "aspect_type", "dist_to_road", "ambient_temperature",
    "ghi", "humidity", "wind_speed", "wind_direction", "dt_wind",
]

NUMERIC_FEATURE_COLS = [
    "longitude", "latitude", "elevation", "area", "slope", "curvature", "aspect",
    "dist_to_road", "ambient_temperature", "ghi", "humidity", "wind_speed", "wind_direction",
]

CATEGORICAL_TERRAIN_COLS = ["slope_type", "curvature_type", "aspect_type"]
CATEGORICAL_FEATURE_COLS = CATEGORICAL_TERRAIN_COLS + ["dt_wind"]

ANGULAR_COLS = ["aspect", "wind_direction"]

EXCLUDED_COLS = [
    "solar_aptitude", "solar_aptitude_rounded",   # fuga directa: definen el target por binning
    "capacity", "optimal_tilt", "pv_potential",   # resultado posterior a construir la planta
    "operational_status",                          # posterior a la decisión de inversión
    "OBJECTID", "code", "plant_name",              # identificadores sin valor predictivo
]

PENDING_COLS = ["size", "country"]  # analizadas en el EDA, no incorporadas todavía como predictoras

missing_expected = [c for c in FEATURE_COLS + [TARGET_COL] if c not in df.columns]
if missing_expected:
    print("ATENCIÓN: columnas de features/target no encontradas:", missing_expected)
else:
    print("Todas las columnas de FEATURE_COLS y TARGET_COL están presentes en el dataset.")
    print(f"FEATURE_COLS: {len(NUMERIC_FEATURE_COLS)} numéricas + {len(CATEGORICAL_FEATURE_COLS)} "
          f"categóricas = {len(FEATURE_COLS)} totales.")

# Verificación de completitud: cada columna del dataset debe caer en exactamente una categoría
# (feature, excluida, pendiente o target); si algo queda fuera, se está pasando por alto sin decisión.
todas_categorizadas = set(FEATURE_COLS) | set(EXCLUDED_COLS) | set(PENDING_COLS) | {TARGET_COL}
sin_categorizar = set(EXPECTED_COLUMNS) - todas_categorizadas
if sin_categorizar:
    print("ATENCIÓN: columnas sin categorizar (ni feature, ni excluida, ni pendiente, ni target):",
          sin_categorizar)
else:
    print(f"Categorización completa: {len(FEATURE_COLS)} features + {len(EXCLUDED_COLS)} excluidas + "
          f"{len(PENDING_COLS)} pendientes + 1 target = {len(todas_categorizadas)} columnas "
          f"(de {len(EXPECTED_COLUMNS)} totales).")


## 3. Inspección inicial

In [ ]:
print(f"Dimensiones del dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas\n")
df.info()


In [ ]:
# Tipos de dato por columna y ejemplo de valor
resumen_tipos = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "ejemplo": df.iloc[0],
    "n_unicos": df.nunique(),
})
resumen_tipos


## 4. Calidad de datos

### 4.1 Valores nulos

In [ ]:
nulos = df.isnull().sum().sort_values(ascending=False)
pct_nulos = (nulos / len(df) * 100).round(2)
tabla_nulos = pd.DataFrame({"n_nulos": nulos, "pct_nulos": pct_nulos})
tabla_nulos = tabla_nulos[tabla_nulos["n_nulos"] > 0]

if tabla_nulos.empty:
    print("No se encontraron valores nulos en el dataset.")
else:
    display(tabla_nulos)

    fig, ax = plt.subplots(figsize=(8, max(1.8, 0.6 * len(tabla_nulos))))

    bars = ax.barh(
        tabla_nulos.index,
        tabla_nulos["pct_nulos"],
        color="#3b6fa0",
        height=0.5,
        zorder=3,
    )

    for bar, pct, n in zip(bars, tabla_nulos["pct_nulos"], tabla_nulos["n_nulos"]):
        ax.text(
            bar.get_width() + 1.5,
            bar.get_y() + bar.get_height() / 2,
            f"{pct:.2f}%  (n={int(n):,})",
            va="center", ha="left",
            fontsize=10, color="#333333",
        )

    ax.set_xlim(0, 100)
    ax.set_xlabel("% de valores nulos")
    ax.set_ylabel("")
    ax.set_title("Porcentaje de valores nulos por columna", fontsize=12, fontweight="bold", pad=12)

    n_total_cols = df.shape[1]
    n_con_nulos = len(tabla_nulos)
    ax.text(
        0.98, 1.12,
        f"{n_con_nulos} de {n_total_cols} columnas presentan valores nulos",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=9, color="#666666", style="italic",
    )

    ax.grid(axis="x", linestyle="--", alpha=0.4, zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(left=False)

    plt.tight_layout()
    plt.savefig("fig_nulos.png", dpi=200, bbox_inches="tight")
    plt.show()

### 4.2 Filas y códigos duplicados

In [ ]:
n_dup_filas = df.duplicated().sum()
n_dup_codigo = df.duplicated(subset=["code"]).sum() if "code" in df.columns else np.nan

print(f"Filas completamente duplicadas: {n_dup_filas}")
print(f"Códigos de planta duplicados:   {n_dup_codigo}")

if n_dup_codigo and n_dup_codigo > 0:
    display(df[df.duplicated(subset=["code"], keep=False)].sort_values("code").head(10))


### 4.3 Verificación y corrección de tipos numéricos

`pandas.read_excel` puede inferir alguna columna numérica como tipo `object` si contiene algún valor no
numérico o mal formado. Esta celda fuerza la conversión a numérico de las columnas de `FEATURE_COLS` que
deban serlo, y registra cuántos valores nuevos quedan como nulos por esa conversión, para no perder ese
diagnóstico silenciosamente.

In [ ]:
NUMERIC_COLS_TO_CHECK = NUMERIC_FEATURE_COLS + [
    "capacity", "optimal_tilt", "solar_aptitude", "solar_aptitude_rounded", "pv_potential",
]
NUMERIC_COLS_TO_CHECK = [c for c in NUMERIC_COLS_TO_CHECK if c in df.columns]

conversion_log = {}
for col in NUMERIC_COLS_TO_CHECK:
    if not pd.api.types.is_numeric_dtype(df[col]):
        antes_na = df[col].isna().sum()
        df[col] = pd.to_numeric(df[col], errors="coerce")
        despues_na = df[col].isna().sum()
        conversion_log[col] = despues_na - antes_na

if conversion_log:
    print("Columnas convertidas a numérico (nuevos NaN introducidos por valores no numéricos):")
    display(pd.DataFrame.from_dict(conversion_log, orient="index", columns=["nuevos_NaN_por_conversion"]))
else:
    print(f"Las {len(NUMERIC_COLS_TO_CHECK)} columnas numéricas verificadas ya tenían dtype numérico.")


### 4.4 Validación de rangos lógicos / físicos

Se verifica que cada variable respete su rango físicamente válido. Cualquier violación indica errores de
captura o de unidades que deben tratarse antes de modelar.

In [ ]:
rango_esperado = {
    "latitude":       (-90, 90),
    "longitude":      (-180, 180),
    "slope":          (0, 90),
    "aspect":         (0, 360),
    "wind_direction": (0, 360),
    "humidity":       (0, 100),
    "elevation":      (-430, 8850),   # Mar Muerto -> Everest, margen razonable
    "wind_speed":     (0, 60),        # m/s, margen amplio para vientos extremos
    "dist_to_road":   (0, None),
    "ghi":            (0, None),
    "area":           (0, None),
}

violaciones = []
for col, (lo, hi) in rango_esperado.items():
    if col not in df.columns:
        continue
    mask = pd.Series(False, index=df.index)
    if lo is not None:
        mask |= df[col] < lo
    if hi is not None:
        mask |= df[col] > hi
    n_viol = mask.sum()
    if n_viol > 0:
        violaciones.append({"columna": col, "rango_valido": f"[{lo}, {hi}]", "n_violaciones": n_viol,
                             "pct": round(n_viol / len(df) * 100, 3)})

if violaciones:
    display(pd.DataFrame(violaciones))
else:
    print("Todas las columnas verificadas están dentro de su rango físico esperado.")


### 4.5 Consistencia entre variable numérica y su categoría asociada

In [ ]:
# Ej.: 'slope' (numérica) vs 'slope_type' (categórica) deben ser coherentes.
# Se muestra la distribución de la variable numérica por cada categoría, para detectar
# inconsistencias de etiquetado entre la variable continua y su categoría asociada.
pares_num_cat = [("slope", "slope_type"), ("curvature", "curvature_type"), ("aspect", "aspect_type")]

for num_col, cat_col in pares_num_cat:
    if num_col in df.columns and cat_col in df.columns:
        print(f"\n--- {num_col} agrupado por {cat_col} ---")
        display(df.groupby(cat_col)[num_col].describe()[["count", "mean", "min", "max"]])


### 4.6 Consistencia del target

In [ ]:
print("Valores únicos en el target:", df[TARGET_COL].dropna().unique())

inesperados = set(df[TARGET_COL].dropna().unique()) - {"Alta", "Media", "Baja"}
print("\n¿Hay valores inesperados fuera de {Alta, Media, Baja}?")
print(inesperados if inesperados else "Ninguno: las 3 categorías esperadas están presentes.")


## 5. Análisis univariado: variable objetivo (`solar_aptittude_class`)

La variable objetivo agrupa cada planta en una de tres categorías ordinales de aptitud solar. La tabla y
el gráfico siguientes muestran su distribución real sobre las 58,978 plantas del dataset.

In [ ]:
conteo_clases = df[TARGET_COL].value_counts().reindex(CLASS_ORDER)
pct_clases = (conteo_clases / conteo_clases.sum() * 100).round(2)

tabla_clases = pd.DataFrame({"n": conteo_clases, "pct": pct_clases})
display(tabla_clases)

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=conteo_clases.index, y=conteo_clases.values, order=CLASS_ORDER,
            hue=conteo_clases.index, palette="viridis", legend=False, ax=ax)
for i, (n, pct) in enumerate(zip(conteo_clases.values, pct_clases.values)):
    ax.text(i, n, f"{n:,}\n({pct}%)", ha="center", va="bottom", fontsize=10)
ax.set_title("Distribución de solar_aptittude_class")
ax.set_xlabel("Clase")
ax.set_ylabel("Número de plantas")
plt.tight_layout()
plt.savefig("fig_distribucion_clases.png", dpi=200, bbox_inches="tight")
plt.show()

ratio_desbalance = conteo_clases["Alta"] / conteo_clases["Baja"]
print(f"\nRatio de desbalance Alta:Baja = {ratio_desbalance:.1f} : 1")


Con una clase minoritaria que representa apenas el 3.21% de los registros, un clasificador que prediga
siempre Alta alcanzaría una accuracy cercana al 75% sin distinguir un solo caso de Baja o Media, un
resultado inútil para el problema que se busca resolver. El desbalance obliga a evaluar los modelos con
métricas que ponderen las clases por igual (F1 macro, recall por clase, matriz de confusión), a
particionar los datos de forma estratificada, y a considerar durante el entrenamiento ponderación de
clases o técnicas de remuestreo como SMOTE o submuestreo de la clase mayoritaria.

## 6. Análisis univariado: variables predictoras numéricas

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 13))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURE_COLS):
    sns.histplot(df[col].dropna(), bins=40, kde=True, ax=axes[i], color="#3b6fa0")
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("")

for j in range(len(NUMERIC_FEATURE_COLS), len(axes)):
    axes[j].axis("off")

plt.suptitle("Distribución de las variables predictoras numéricas", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Tabla resumen estadístico de todas las features numéricas
df[NUMERIC_FEATURE_COLS].describe().T.assign(
    skewness=df[NUMERIC_FEATURE_COLS].skew(),
    missing_pct=(df[NUMERIC_FEATURE_COLS].isna().mean() * 100).round(2),
)


Las variables numéricas muestran formas muy distintas entre sí. `dist_to_road` (asimetría 48.74) y `area`
(72.62, con 27.69% de nulos) tienen colas extremadamente largas: la mediana de `area` es de 8,067 unidades
pero el máximo supera los 234 millones, evidencia de que unas pocas plantas de escala muy grande dominan
la parte superior de la distribución. `slope` (3.98), `wind_speed` (3.56) y `elevation` (3.18) también
están sesgadas a la derecha, mientras que `latitude` (-1.96) y `humidity` (-1.49) lo están a la izquierda.
`aspect`, pese a incluir el valor centinela -1 para terreno plano, resulta casi simétrica (0.035), lo cual
es consistente con que las direcciones cardinales están representadas de forma relativamente uniforme.

## 7. Variables categóricas (`slope_type`, `curvature_type`, `aspect_type`, `dt_wind`)

In [ ]:
for col in CATEGORICAL_FEATURE_COLS:
    if col not in df.columns:
        continue
    n_cat = df[col].nunique()
    print(f"\n{col}: {n_cat} categorías únicas")
    display(df[col].value_counts().to_frame("count"))

fig, axes = plt.subplots(1, len(CATEGORICAL_FEATURE_COLS), figsize=(5 * len(CATEGORICAL_FEATURE_COLS), 4.5))
for ax, col in zip(axes, CATEGORICAL_FEATURE_COLS):
    if col not in df.columns:
        continue
    top = df[col].value_counts()
    sns.barplot(x=top.values, y=top.index, hue=top.index, palette="viridis", legend=False, ax=ax)
    ax.set_title(f"Categorías: {col}")
plt.tight_layout()
plt.show()


`slope_type` está dominada por terreno plano o casi plano (46,766 plantas, 79.3% del dataset), seguido de
pendiente suave (8,523, 14.5%); las categorías de pendiente fuerte, muy fuerte o escarpada suman apenas
783 plantas (1.3%). `curvature_type` muestra un patrón similar: 81.1% de las superficies son planas o
intermedias. `aspect_type` está más repartida entre las ocho orientaciones cardinales (entre 4,906 y 8,335
plantas cada una), más 1,105 plantas (1.9%) marcadas como `Flat`, que coinciden exactamente con las 1,105
violaciones de rango detectadas en `aspect` (el valor centinela -1). `dt_wind`, en cambio, está fuertemente
concentrada en tres direcciones: `Southwest`, `West` y `Southeast` suman el 73.2% del dataset, mientras
que `Northeast`, `North` y `Northwest` representan en conjunto apenas el 0.6%.

## 8. Análisis geográfico y sesgo de cobertura

Dado que el objetivo final del proyecto es generalizar a cualquier ubicación nueva, es crítico cuantificar
qué tan sesgada está la cobertura geográfica del dataset: un modelo entrenado con datos concentrados en
pocos países puede generalizar mal fuera de esas zonas.

In [ ]:
n_total = len(df)
n_antes = len(df.dropna(subset=["latitude", "longitude", TARGET_COL]))

for col in ["latitude", "longitude"]:
    print(f"{col}: dtype = {df[col].dtype}")
    coerced = pd.to_numeric(df[col], errors="coerce")
    no_convertibles = coerced.isna() & df[col].notna()
    n_no_convertibles = int(no_convertibles.sum())
    print(f"{col}: {n_no_convertibles} valores no convertibles a número de {n_total}")
    if n_no_convertibles > 0:
        print(f"muestra de valores no numéricos en {col}:", df.loc[no_convertibles, col].unique()[:15])
    df[col] = coerced
    print()

n_despues = len(df.dropna(subset=["latitude", "longitude", TARGET_COL]))
print(f"Filas utilizables para el mapa antes de la corrección: {n_antes} de {n_total} "
      f"({n_antes / n_total * 100:.2f}%)")
print(f"Filas utilizables para el mapa después de la corrección: {n_despues} de {n_total} "
      f"({n_despues / n_total * 100:.2f}%)")


In [ ]:
fraccion_utilizable = n_despues / n_total

if fraccion_utilizable < 0.9:
    print(f"Solo el {fraccion_utilizable * 100:.2f}% de las filas tiene latitude/longitude/target "
          f"utilizables; no se genera el mapa hasta resolver la causa de la pérdida de datos.")
else:
    df_geo = df.dropna(subset=["latitude", "longitude", TARGET_COL])

    fig = px.scatter_geo(
        df_geo,
        lat="latitude", lon="longitude",
        color=TARGET_COL,
        category_orders={TARGET_COL: CLASS_ORDER},
        color_discrete_sequence=["#c0392b", "#e0a72b", "#2f6f4f"],
        hover_name="country" if "country" in df.columns else None,
        opacity=0.5,
        title="Distribución global de plantas solares, color = solar_aptittude_class",
    )
    fig.update_layout(height=550, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()


In [ ]:
if "country" in df.columns:
    conteo_paises = df["country"].value_counts()
    top10 = conteo_paises.head(10)
    pct_top10 = top10.sum() / len(df) * 100

    print(f"Países únicos: {df['country'].nunique()}")
    print(f"Los 10 países con más registros concentran el {pct_top10:.1f}% del dataset:")
    display(top10.to_frame("n_plantas"))

    # Proporción de cada clase de aptitud por país (top 10), para ver si el desbalance
    # de clases también varía geográficamente
    tabla_pais_clase = (
        df[df["country"].isin(top10.index)]
        .groupby("country")[TARGET_COL]
        .value_counts(normalize=True)
        .unstack()
        .reindex(columns=CLASS_ORDER)
        .reindex(top10.index)
    )
    display((tabla_pais_clase * 100).round(1))
else:
    print("Columna 'country' no encontrada.")


Los diez países con más registros (China, Japón, Estados Unidos, Alemania, India, España, Reino Unido,
Corea del Sur, Francia y Brasil) concentran el 79.8% de las 58,978 plantas, aunque el dataset cubre 183
países en total. La composición de clases varía de forma marcada entre ellos: China, Japón, India, Corea
del Sur y Brasil no registran ninguna planta Baja, y Estados Unidos tiene solo una entre sus 7,224
plantas. En cambio, Alemania (4.6%), España (8.2%), Reino Unido (3.2%) y Francia (10.4%) concentran casi
toda la representación de esa clase entre los países dominantes: la clase minoritaria del problema es, en
la práctica, casi exclusivamente europea dentro de este subconjunto. Un split aleatorio de entrenamiento y
prueba dejaría que el modelo memorice esta asociación entre país y clase en lugar de aprender una relación
transferible a partir del terreno y el clima; una validación cruzada espacial, agrupando por país o
continente, ofrece una estimación más realista de la capacidad de generalizar a ubicaciones nuevas.

La alta cardinalidad de `country` (183 categorías) hace poco práctico incorporarlo directamente como
predictora categórica sin alguna forma de agregación. Codificarlo por región o continente, o dejarlo fuera
del conjunto de predictoras en favor de que el modelo aprenda únicamente de terreno y clima, son las
alternativas más razonables a evaluar en la etapa de modelado.

## 9. Variables angulares (`aspect`, `wind_direction`)

Estas variables son **circulares** (0° y 359° representan direcciones casi idénticas), por lo que un
histograma lineal estándar puede ser engañoso, y no deben alimentarse directamente a un modelo como
variables numéricas lineales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={"projection": "polar"})

for ax, col in zip(axes, ANGULAR_COLS):
    data = np.deg2rad(df[col].dropna())
    ax.hist(data, bins=36, color="#8a4fbf", alpha=0.75)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(col)

plt.suptitle("Distribución circular de variables angulares", y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
# Demostración del encoding cíclico (seno/coseno) que se usará en la etapa de modelado
for col in ANGULAR_COLS:
    rad = np.deg2rad(df[col])
    df[f"{col}_sin"] = np.sin(rad)
    df[f"{col}_cos"] = np.cos(rad)

df[ANGULAR_COLS + [f"{ANGULAR_COLS[0]}_sin", f"{ANGULAR_COLS[0]}_cos"]].head()


## 10. Chequeo de fuga de datos

`solar_aptitude`, `solar_aptitude_rounded`, `capacity`, `optimal_tilt`, `pv_potential` y
`operational_status` no forman parte del conjunto de predictoras, pero conviene documentar su relación con
el target: tanto para justificar la exclusión con evidencia, como para usarlas como variables de contexto
en la interpretación de resultados, nunca como predictoras.

In [ ]:
cols_leakage_numeric = [c for c in ["solar_aptitude", "solar_aptitude_rounded", "capacity",
                                     "optimal_tilt", "pv_potential"] if c in df.columns]

for col in cols_leakage_numeric:
    print(f"\n--- {col} por clase de {TARGET_COL} ---")
    display(df.groupby(TARGET_COL)[col].describe()[["count", "mean", "std", "min", "max"]].reindex(CLASS_ORDER))

fig, axes = plt.subplots(1, len(cols_leakage_numeric), figsize=(5 * len(cols_leakage_numeric), 4.5))
if len(cols_leakage_numeric) == 1:
    axes = [axes]
for ax, col in zip(axes, cols_leakage_numeric):
    sns.boxplot(data=df, x=TARGET_COL, y=col, order=CLASS_ORDER, hue=TARGET_COL,
                palette="viridis", legend=False, ax=ax)
    ax.set_title(f"{col} por clase")
plt.tight_layout()
plt.show()


`solar_aptitude` y `solar_aptitude_rounded` separan las tres clases de forma exacta: sus rangos por clase
no se superponen (Baja hasta 0.400, Media entre 0.4 y 0.6, Alta desde 0.6), lo que confirma que son la
misma variable subyacente que define el target por binning y justifica su exclusión total. `capacity`, en
cambio, no muestra una diferencia real entre clases (medias de 44.4, 37.8 y 39.4 MW para Baja, Media y
Alta, con desviaciones estándar de más de 300 en los tres casos): su exclusión se sostiene por ser
información posterior a la inversión, no por evidencia de fuga estadística. `optimal_tilt` presenta una
diferencia más consistente aunque modesta entre clases (medias de 33.7°, 33.1° y 30.6°), y `pv_potential`
tampoco distingue claramente las clases (medias de 3.88, 3.92 y 3.76 kWh/kWp). En conjunto, la única fuga
directa y contundente proviene de `solar_aptitude` y su versión redondeada; las otras tres variables se
excluyen por diseño, por ser posteriores a la decisión de inversión, y no porque los datos muestren que
filtran información del target.

In [ ]:
if "operational_status" in df.columns:
    tabla_estado = pd.crosstab(df["operational_status"], df[TARGET_COL], normalize="index")[CLASS_ORDER] * 100
    display(tabla_estado.round(1))
    print("\n'operational_status' se excluye de todas formas por ser posterior a la decisión de inversión.")


## 11. Análisis bivariado: variables predictoras vs. `solar_aptittude_class`

Como el target es categórico (no continuo), la relación con cada feature numérica se visualiza con
boxplots por clase y se evalúa formalmente con el test de Kruskal-Wallis (alternativa no paramétrica al
ANOVA de un factor, no asume normalidad) para las diferencias de mediana entre las 3 clases.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURE_COLS):
    sns.boxplot(data=df, x=TARGET_COL, y=col, order=CLASS_ORDER, hue=TARGET_COL,
                palette="viridis", legend=False, ax=axes[i], fliersize=1.5)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("")

for j in range(len(NUMERIC_FEATURE_COLS), len(axes)):
    axes[j].axis("off")

plt.suptitle("Distribución de cada feature numérica por clase de aptitud solar", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
resultados_kw = []
for col in NUMERIC_FEATURE_COLS:
    grupos = [df.loc[df[TARGET_COL] == c, col].dropna() for c in CLASS_ORDER]
    if not all(len(g) > 0 for g in grupos):
        continue
    try:
        stat, p = kruskal(*grupos)
        resultados_kw.append({"feature": col, "kruskal_H": round(stat, 2), "p_valor": p})
    except ValueError as e:
        print(f"No se pudo calcular Kruskal-Wallis para '{col}': {e}")
        resultados_kw.append({"feature": col, "kruskal_H": np.nan, "p_valor": np.nan})

tabla_kw = pd.DataFrame(resultados_kw).sort_values("kruskal_H", ascending=False, na_position="last")
tabla_kw["significativo_0.05"] = tabla_kw["p_valor"] < 0.05
tabla_kw


Las 13 features numéricas resultan estadísticamente significativas (p < 0.05) en el test de
Kruskal-Wallis frente a las tres clases, aunque este resultado debe leerse con cautela: con casi 59,000
observaciones, incluso diferencias muy pequeñas entre grupos alcanzan significancia estadística, por lo
que la magnitud del estadístico H importa más que el simple hecho de que p sea menor a 0.05. `longitude`
domina por un margen amplio (H=12,890.78), muy por encima de `slope` (H=3,712.68), `dist_to_road`
(H=2,605.13) y `latitude` (H=2,175.86). Esa magnitud es consistente con el patrón geográfico descrito
antes: buena parte de la diferencia entre clases a lo largo de la longitud probablemente refleja la
composición de países del dataset, con la clase Baja concentrada en un puñado de países europeos, más que
una relación física directa entre longitud y aptitud solar. En el otro extremo, `curvature` (H=19.54) y
`aspect` (H=22.41) son las variables con menor poder discriminante, aunque siguen siendo significativas.

## 12. Correlación entre features y multicolinealidad (VIF)

La correlación aquí se calcula solo entre las features predictoras numéricas, no con el target (que es
categórico y cuya relación con cada feature ya se evaluó mediante Kruskal-Wallis). El VIF mide redundancia
entre predictoras; un valor mayor a 10 suele indicar multicolinealidad significativa que puede requerir
eliminar o combinar variables, o aplicar regularización antes de modelar.

In [ ]:
corr_matrix = df[NUMERIC_FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Matriz de correlación: features numéricas predictoras")
plt.tight_layout()
plt.savefig("fig_correlacion.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
vif_data = df[NUMERIC_FEATURE_COLS].dropna()

vif_df = pd.DataFrame({
    "feature": NUMERIC_FEATURE_COLS,
    "VIF": [variance_inflation_factor(vif_data.values, i) for i in range(len(NUMERIC_FEATURE_COLS))],
}).sort_values("VIF", ascending=False)

vif_df


Ninguna de las 13 features numéricas presenta multicolinealidad relevante: el VIF más alto corresponde a
`ghi` (4.29), seguido de `latitude` (3.40) y `ambient_temperature` (3.07); el resto se ubica por debajo de
2.4. Todos los valores quedan muy por debajo del umbral de 10, por lo que no hay evidencia de redundancia
severa entre predictoras que obligue a excluir o combinar variables antes de entrenar un modelo lineal.

## 13. Detección de outliers (método IQR)

In [ ]:
def resumen_outliers_iqr(serie, factor=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - factor * iqr, q3 + factor * iqr
    outliers = serie[(serie < lo) | (serie > hi)]
    return {
        "n_outliers": len(outliers),
        "pct_outliers": round(len(outliers) / serie.notna().sum() * 100, 2),
        "limite_inferior": round(lo, 3),
        "limite_superior": round(hi, 3),
    }

outlier_summary = pd.DataFrame({
    col: resumen_outliers_iqr(df[col].dropna()) for col in NUMERIC_FEATURE_COLS
}).T.sort_values("pct_outliers", ascending=False)

outlier_summary


El método IQR marca a `curvature` como la variable con mayor proporción de outliers (18.95%), seguida de
`area` (13.53%, sobre las 42,649 observaciones no nulas) y `dist_to_road` (12.78%). En el caso de
`curvature`, esto es en gran parte un artefacto del método: la variable está tan concentrada alrededor de
cero que su rango intercuartílico es minúsculo, y cualquier desviación moderada queda marcada como
atípica. `area` y `dist_to_road`, en cambio, sí reflejan una cola larga genuina: 1,261 plantas superan por
sí solas el millón de unidades de área. `longitude`, `aspect` y `wind_direction` no presentan ningún
outlier bajo este criterio, consistente con ser variables acotadas por diseño físico.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 13))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURE_COLS):
    sns.boxplot(x=df[col], ax=axes[i], color="#e07b39")
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("")

for j in range(len(NUMERIC_FEATURE_COLS), len(axes)):
    axes[j].axis("off")

plt.suptitle("Boxplots de todas las variables predictoras (detección visual de outliers)", y=1.01)
plt.tight_layout()
plt.show()


## 14. Segmentación adicional (`size`, banda climática)

Se exploran dos ejes adicionales de segmentación: `size`, una variable categórica (`Small` / `Medium` /
`Big`) que todavía no forma parte del conjunto de predictoras, y la banda climática derivada de
`latitude`.

In [ ]:
if "size" in df.columns:
    print(f"'size': {df['size'].nunique()} categorías únicas")
    display(df["size"].value_counts().to_frame("count"))

    if "operational_status" in df.columns:
        print("\n'size' por operational_status (para verificar si describe el terreno o el proyecto ya "
              "construido, igual que se hizo con 'area'):")
        tabla_size_estado = pd.crosstab(df["operational_status"], df["size"], normalize="index") * 100
        display(tabla_size_estado.round(1))

    print("\n'size' por clase de aptitud solar:")
    tabla_size_clase = pd.crosstab(df["size"], df[TARGET_COL], normalize="index")[CLASS_ORDER] * 100
    display(tabla_size_clase.round(1))
else:
    print("Columna 'size' no encontrada.")


`size` se reparte de forma desigual: 41,069 plantas (69.6%) son `Small`, 13,208 (22.4%) `Medium` y 4,701
(8.0%) `Big`. Su distribución varía con el estado operacional de la planta: entre las categorías con
mayor volumen de datos, `operating` (n=47,538) tiene 7.9% de plantas `Big` y 24.5% `Medium`, mientras que
`pre-construction` (n=6,246) tiene una proporción similar de `Big` (8.2%) pero muchas menos `Medium`
(9.7%, frente a 82.1% `Small`). Esta variación, aunque no extrema, sugiere que `size` codifica al menos
parcialmente el estado de avance del proyecto y no únicamente una característica del terreno disponible
antes de invertir. Frente al target, la relación es más modesta: la categoría `Big` tiene la mayor
proporción de `Media` (30.2%) y la menor de `Baja` (1.7%), mientras que `Small` tiene la mayor proporción
tanto de `Alta` (78.3%) como de `Baja` (3.6%).

### Banda climática (latitud)

In [ ]:
bins_lat = [-90, -66.5, -23.5, 0, 23.5, 66.5, 90]
labels_lat = ["Polar S", "Templada S", "Trop. S", "Trop. N", "Templada N", "Polar N"]
df["banda_climatica"] = pd.cut(df["latitude"], bins=bins_lat, labels=labels_lat)

tabla_banda = pd.crosstab(df["banda_climatica"], df[TARGET_COL], normalize="index")[CLASS_ORDER] * 100

fig, ax = plt.subplots(figsize=(9, 5))
tabla_banda.plot(kind="bar", stacked=True, color=["#c0392b", "#e0a72b", "#2f6f4f"], ax=ax)
ax.set_title("Proporción de clases de aptitud solar por banda climática (latitud)")
ax.set_ylabel("% de plantas")
ax.legend(title=TARGET_COL)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

tabla_banda.round(1)


La banda tropical norte concentra la mayor proporción de aptitud Alta (87.2%) y la menor de Baja (0.5%),
mientras que las bandas templadas y la tropical sur muestran una mezcla más pareja entre Media y Alta, con
Media superando el 38% en ambas bandas del hemisferio sur. Ninguna planta del dataset cae en las bandas
polares, consistente con que la latitud observada va de -41.6° a 59.9°.

## 15. Conclusiones

El dataset resultó, en general, limpio: no se encontraron filas duplicadas ni códigos de planta repetidos
entre las 58,978 observaciones, y ninguna columna presentó un formato numérico inconsistente tras la
carga. La única ausencia de datos relevante está en `area`, con 16,329 valores nulos (27.69%). La única
violación de rango detectada fue en `aspect`, con 1,105 registros (1.87%) fuera de [0°, 360°]; coincide
exactamente con las 1,105 plantas etiquetadas como `Flat` en `aspect_type`, lo que confirma que el valor
-1 es un centinela deliberado para terreno sin orientación definida y no un error de captura.

La variable objetivo está fuertemente desbalanceada: 75.10% de las plantas son Alta, 21.69% Media y apenas
3.21% Baja, una razón de 23.4 a 1 entre la clase mayoritaria y la minoritaria. Cualquier modelo evaluado
únicamente por accuracy puede parecer competente sin distinguir un solo caso de Baja; la comparación de
modelos en las siguientes etapas debe apoyarse en F1 macro, recall por clase y matrices de confusión,
además de una partición estratificada del conjunto de datos.

Entre las variables numéricas, `area` (asimetría 72.62) y `dist_to_road` (48.74) tienen las colas más
largas, seguidas de `slope` (3.98), `wind_speed` (3.56) y `elevation` (3.18); `latitude` (-1.96) y
`humidity` (-1.49) están sesgadas en la dirección opuesta. Las variables categóricas de terreno están
dominadas por una sola categoría: 79.3% de las plantas están en terreno plano o casi plano (`slope_type`)
y 81.1% en superficies planas o intermedias (`curvature_type`). `dt_wind`, la dirección predominante del
viento, concentra el 73.2% de los registros en solo tres de sus ocho categorías (`Southwest`, `West`,
`Southeast`).

El hallazgo más importante del análisis geográfico es que la clase Baja está casi enteramente concentrada
en un puñado de países europeos: Alemania, España, Reino Unido y Francia son los únicos países, entre los
diez con más registros, que registran una proporción no marginal de esa clase (entre 3.2% y 10.4%),
mientras que China, Japón, India, Corea del Sur y Brasil no tienen ninguna planta Baja, y Estados Unidos
tiene una sola entre 7,224. Esto, sumado a que `longitude` obtuvo el estadístico de Kruskal-Wallis más
alto de todo el análisis (H=12,890.78, muy por encima del resto), sugiere que un modelo entrenado sin
cuidado podría aprender a asociar región geográfica con clase de aptitud en lugar de una relación física
entre terreno, clima y aptitud. Una validación cruzada espacial, agrupando por país o continente en lugar
de un split aleatorio, es la forma más honesta de estimar qué tan bien generalizaría el modelo a
ubicaciones fuera de los 183 países representados, de los cuales solo diez concentran el 79.8% del
dataset.

El chequeo de fuga de datos confirma que `solar_aptitude` y `solar_aptitude_rounded` separan las tres
clases sin superposición (son, de hecho, la misma variable de la que se deriva el target por binning),
pero no encontró evidencia estadística de fuga en `capacity` (medias casi idénticas entre clases: 44.4,
37.8 y 39.4 MW) ni en `pv_potential` (3.88, 3.92 y 3.76 kWh/kWp); `optimal_tilt` mostró una diferencia más
consistente pero moderada (33.7°, 33.1° y 30.6°). Su exclusión del conjunto de predictoras se sostiene,
entonces, por ser información posterior a la decisión de inversión, y no porque los datos muestren que
filtran directamente el target.

Las 13 features numéricas resultaron estadísticamente significativas frente al target en el test de
Kruskal-Wallis, aunque con un tamaño de muestra de casi 59,000 observaciones incluso diferencias pequeñas
alcanzan significancia: la magnitud del estadístico importa más que el umbral de p. Ninguna combinación de
predictoras presenta multicolinealidad relevante (el VIF más alto es 4.29, para `ghi`, muy por debajo del
umbral habitual de 10), por lo que no hay razón para excluir o combinar variables antes de entrenar un
modelo lineal por redundancia entre predictoras.

La detección de outliers por rango intercuartílico señala a `curvature` (18.95%), `area` (13.53%) y
`dist_to_road` (12.78%) como las variables con mayor proporción de valores atípicos. En `curvature`, esto
responde más al método que a los datos: la variable está tan concentrada cerca de cero que su rango
intercuartílico es minúsculo y cualquier desviación moderada queda marcada como atípica. En `area` y
`dist_to_road`, en cambio, la cola es genuinamente larga: 1,261 plantas superan por sí solas el millón de
unidades de área. Esto favorece modelos robustos a outliers (árboles de decisión, ensambles) sobre una
regresión lineal sin tratamiento previo de estas colas.

`size` (`Small` 69.6%, `Medium` 22.4%, `Big` 8.0%) varía con el estado operacional de la planta lo
suficiente como para sospechar que codifica, al menos en parte, el tamaño del proyecto ya construido y no
solo el terreno disponible antes de invertir; su incorporación como predictora debería depender de si
aporta información más allá de `area` y `capacity`. La segmentación por banda climática confirma el
patrón esperado: la banda tropical norte concentra 87.2% de aptitud Alta, frente a un 55-58% en las
bandas templadas y tropical sur. Por último, conviene recordar la limitación estructural de todo este
análisis: el dataset solo contiene sitios donde ya existe una planta construida, no ejemplos de sitios
descartados o nunca evaluados. Cualquier modelo entrenado aquí aprende a distinguir aptitud entre lugares
donde alguien ya decidió invertir, y su uso para evaluar ubicaciones genuinamente nuevas descansa en el
supuesto, no verificado por los datos, de que terreno y clima similares producen aptitud similar.